In [10]:
import numpy as np
import pymysql
import pandas as pd
# ==================== 1. 범용 경로 설정 ====================
import sys
import os
from typing import Union
from pathlib import Path

def setup_universal_paths():
    """
    어떤 PC에서도 작동하는 범용 경로 설정
    DATA 폴더를 자동으로 찾아 경로 추가
    """
    current = Path.cwd()

    # 상위 폴더를 탐색하며 DATA 폴더 찾기
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            # 프로젝트 루트와 DATA 폴더 모두 추가
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            if str(data_folder) not in sys.path:
                sys.path.insert(0, str(data_folder))

            print("=" * 70)
            print("📁 경로 설정 완료")
            print("=" * 70)
            print(f"✓ 프로젝트 루트: {parent}")
            print(f"✓ DATA 폴더:    {data_folder}")
            print(f"✓ 현재 위치:     {current}")
            print(f"✓ 운영체제:      {os.name}")
            print("=" * 70 + "\n")

            return {
                'project_root': parent,
                'data_folder': data_folder,
                'current': current
            }

    # 못 찾으면 에러
    raise FileNotFoundError(
        f"❌ DATA 폴더를 찾을 수 없습니다.\n"
        f"현재 위치: {current}\n"
        f"상위 폴더에 DATA 폴더가 있는지 확인하세요."
    )

# 경로 설정 실행
try:
    paths = setup_universal_paths()
except FileNotFoundError as e:
    print(e)
    print("\n대안: 수동으로 경로를 설정하세요.")
    # sys.path.insert(0, "여기에_프로젝트_루트_경로_입력")
    sys.exit(1)

📁 경로 설정 완료
✓ 프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
✓ DATA 폴더:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
✓ 현재 위치:     C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Korea_Market\analysis\한국기업_매출예측
✓ 운영체제:      nt



In [11]:
# ==================== 2. 필요한 모듈 import ====================
try:
    # 예측 함수 import (파일명 확인 필요!)
    from universal_ts_forecast_function import (
        forecast_one_from_pivot_inline,
        monitor_memory_usage
    )
    from stock_invest_function import fetch_table_data

    print("✓ 예측 모듈 import 성공")

except ImportError as e:
    print(f"❌ 모듈 import 실패: {e}")
    print("\n확인 사항:")
    print("1. DATA 폴더에 'universal_ts_forecast_function.py' 파일이 있는가?")
    print("2. DATA 폴더에 'stock_invest_function.py' 파일이 있는가?")
    print("\n파일명이 다르다면 위 import 문을 수정하세요.")
    sys.exit(1)

from universal_ts_forecast_function import (
    ensure_datetime_index_df,
    forecast_sarima,
    forecast_ets,
    forecast_theta,
    infer_freq_alias,
    seasonal_periods_from_freq
)

from DATA.stock_invest_function import *

def get_connection(db_info: dict):
    conn = pymysql.connect(
        host=db_info["host"],
        port=int(db_info["port"]),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info.get("db", db_info.get("database")),
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor,
    )
    return conn

def extract_revenue_from_fs_df(fs_df: pd.DataFrame,
                               ticker: str,
                               keyword: str = "매출") -> pd.DataFrame:
    """
    korea_fs_data 전체 df(fs_df)에서 특정 ticker의 매출 관련 시계열만 뽑아오는 함수.

    Parameters
    ----------
    fs_df : pd.DataFrame
        fetch_table_data(db_info, "korea_fs_data") 로 가져온 전체 테이블
    ticker : str
        '005930' 또는 'A005930' 둘 다 허용
    keyword : str
        indicator 에 포함될 키워드 (기본값: '매출')

    Returns
    -------
    pd.DataFrame
        date, revenue, indicator 컬럼을 가진 시계열
    """

    # 1) 심볼 통일: 앞에 A 붙이기
    if ticker.startswith("A"):
        symbol = ticker
    else:
        symbol = "A" + ticker

    # 2) 해당 ticker + 매출 관련 indicator 필터링
    mask_symbol = fs_df["symbol"] == symbol
    mask_ind = fs_df["indicator"].astype(str).str.contains(keyword, na=False)

    sub = fs_df.loc[mask_symbol & mask_ind, ["date", "value", "indicator"]].copy()

    if sub.empty:
        print(f"⚠ {symbol} 에 대해 '{keyword}' 를 포함하는 indicator 가 없습니다.")
        print("   우선 아래 코드를 한 번 실행해서 indicator 목록을 눈으로 확인해 보세요:")
        print("   fs_df[fs_df['symbol']=='A005930']['indicator'].unique()")
        return sub

    # 3) 타입 정리
    sub["date"] = pd.to_datetime(sub["date"], errors="coerce")
    sub = sub.dropna(subset=["date"])

    sub["revenue"] = pd.to_numeric(sub["value"], errors="coerce")

    # 출력 형식 정리
    return sub[["date", "revenue", "indicator"]].sort_values("date").reset_index(drop=True)

def fetch_dart_fs_by_ticker_from_db(
    db_info: dict,
    ticker: Union[str, int],
    table_name: str = "korea_fs_data_from_DART",
    verbose: bool = True,
) -> pd.DataFrame:
    database_name = db_info.get("db") or db_info.get("database")

    if isinstance(ticker, int):
        ticker_str = f"{ticker:06d}"
    else:
        ticker_str = str(ticker).zfill(6)

    # 이 부분만 수정 - DictCursor 제거!
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=database_name,
        charset="utf8mb4",
    )

    try:
        sql = f"""
            SELECT *
            FROM {table_name}
            WHERE ticker = %s
            ORDER BY bsns_year, reprt_code, quarter, account_id
        """
        df = pd.read_sql(sql, conn, params=[ticker_str])

        if verbose:
            print(f"조회 완료: {len(df)}행, {len(df.columns)}개 컬럼")

        return df
    finally:
        conn.close()

def fetch_revenue_by_ticker(
    db_info: dict,
    ticker: Union[str, int],
    table_name: str = "korea_fs_data_from_DART",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    특정 ticker의 매출(Revenue) 데이터를 추출하는 함수

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보 (host, port, user, password, database)
    ticker : str or int
        종목 코드 (예: '005930' 또는 5930)
    table_name : str
        테이블 이름 (기본값: 'korea_fs_data_from_DART')
    verbose : bool
        진행 상황 출력 여부

    Returns:
    --------
    pd.DataFrame
        매출 데이터 (report_date 순으로 정렬)
        컬럼: quarter, account_id, sj_div, sj_nm, account_nm,
              thstrm_nm, thstrm_amount, report_date, ticker, bsns_year, reprt_code
    """
    database_name = db_info.get("db") or db_info.get("database")
    if database_name is None:
        raise KeyError("db_info 안에 'db' 또는 'database' 키가 없습니다!")

    # ticker 정규화
    if isinstance(ticker, int):
        ticker_str = f"{ticker:06d}"
    else:
        ticker_str = str(ticker).zfill(6)

    if verbose:
        print(f"[INFO] 조회 ticker: {ticker_str}")
        print(f"[INFO] 조회 account_id: ['ifrs_Revenue', 'ifrs-full_Revenue']")

    # DB 연결 (DictCursor 제거)
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=database_name,
        charset="utf8mb4",
    )

    try:
        # Revenue 데이터만 조회
        sql = f"""
            SELECT
                quarter,
                account_id,
                sj_div,
                sj_nm,
                account_nm,
                thstrm_nm,
                thstrm_amount,
                report_date,
                ticker,
                bsns_year,
                reprt_code
            FROM {table_name}
            WHERE ticker = %s
                AND account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
            ORDER BY report_date, reprt_code, quarter
        """

        df = pd.read_sql(sql, conn, params=[ticker_str])

        if verbose:
            print(f"[INFO] 조회 완료: {len(df)}행")
            if not df.empty:
                print(f"[INFO] 기간: {df['report_date'].min()} ~ {df['report_date'].max()}")
                print(f"[INFO] account_id 분포:")
                print(df['account_id'].value_counts())

        return df

    finally:
        conn.close()

def adjust_fy_to_q4(df: pd.DataFrame) -> pd.DataFrame:
    """
    FY(연간 누적) 데이터를 순수 Q4로 변환

    Logic:
    ------
    각 연도별로:
    - Q1, Q2, Q3: 원본 그대로 (순수 분기 실적)
    - FY: FY - (Q1 + Q2 + Q3) = 순수 Q4

    Parameters:
    -----------
    df : pd.DataFrame
        매출 데이터 (quarter, bsns_year, thstrm_amount 컬럼 필수)

    Returns:
    --------
    pd.DataFrame
        'quarter' 컬럼이 'Q4'로 변경되고
        'thstrm_amount'가 순수 분기 실적으로 조정된 DataFrame
    """
    result_df = df.copy()

    # 연도별 처리
    for year in result_df['bsns_year'].unique():
        year_mask = result_df['bsns_year'] == year
        year_data = result_df[year_mask]

        # FY 행 찾기
        fy_mask = year_mask & (result_df['quarter'] == 'FY')

        if fy_mask.any():
            # FY 금액
            fy_amount = result_df.loc[fy_mask, 'thstrm_amount'].iloc[0]

            # Q1, Q2, Q3 금액 합계
            q123_mask = year_mask & result_df['quarter'].isin(['Q1', 'H1', 'Q3'])
            q123_sum = result_df.loc[q123_mask, 'thstrm_amount'].sum()

            # 순수 Q4 계산
            pure_q4 = fy_amount - q123_sum

            # FY를 Q4로 변경하고 금액 조정
            result_df.loc[fy_mask, 'quarter'] = 'Q4'
            result_df.loc[fy_mask, 'thstrm_amount'] = pure_q4

            print(f"{year}년: FY({fy_amount:,}) - Q1+Q2+Q3({q123_sum:,}) = Q4({pure_q4:,})")

    return result_df


def get_quarterly_revenue_simple(
    db_info: dict,
    ticker: Union[str, int],
    adjust_q4: bool = True
) -> pd.DataFrame:
    """
    매출 데이터 조회 + Q4 조정을 한번에 수행

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보
    ticker : str or int
        종목 코드
    adjust_q4 : bool
        FY를 순수 Q4로 변환할지 여부 (기본값: True)

    Returns:
    --------
    pd.DataFrame
        순수 분기별 매출 데이터
    """
    import pymysql

    # DB 연결 정보
    database_name = db_info.get("db") or db_info.get("database")

    # ticker 정규화
    if isinstance(ticker, int):
        ticker_str = f"{ticker:06d}"
    else:
        ticker_str = str(ticker).zfill(6)

    # DB 연결
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=database_name,
        charset="utf8mb4",
    )

    try:
        # 매출 데이터 조회
        sql = """
            SELECT *
            FROM korea_fs_data_from_DART
            WHERE ticker = %s
                AND account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
            ORDER BY bsns_year, report_date
        """

        df = pd.read_sql(sql, conn, params=[ticker_str])

        if df.empty:
            print(f"⚠️ {ticker_str} 데이터 없음")
            return df

        print(f"✅ {ticker_str} 매출 데이터 {len(df)}행 조회")

        # Q4 조정
        if adjust_q4:
            df = adjust_fy_to_q4(df)

        return df

    finally:
        conn.close()


def get_all_tickers(db_info: dict) -> list:
    """
    가장 간단한 버전: unique ticker 리스트만 반환

    Parameters:
    -----------
    db_info : dict
        DB 연결 정보

    Returns:
    --------
    list
        unique ticker 리스트 (정렬됨)
    """
    database_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=database_name,
        charset="utf8mb4",
    )

    try:
        sql = """
            SELECT DISTINCT ticker
            FROM korea_fs_data_from_DART
            ORDER BY ticker
        """

        df = pd.read_sql(sql, conn)
        tickers = df['ticker'].tolist()

        print(f"✅ unique ticker {len(tickers)}개 조회 완료")

        return tickers

    finally:
        conn.close()


✓ 예측 모듈 import 성공


In [12]:
# 예시 db_info 채워 넣으신 뒤 테스트
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),  # 노트북에서는 다른 IP일 수 있음
    'port': 3307,
    'database': 'investar'
}

conn = get_connection(db_info)

# 이미 있으신 코드
fs_df = fetch_table_data(db_info, "korea_fs_data")

✅ 'korea_fs_data' 테이블에서 5902708건의 데이터를 가져왔습니다.


In [13]:
all_tickers = get_all_tickers(db_info)

✅ unique ticker 2451개 조회 완료


In [15]:
# =============================================================================
# 분기별 매출 예측 - 빈도 문제 해결 버전 (개선)
# =============================================================================

H = 9   # horizon = 9 steps (9분기)

ticker = "131290"
all_tickers = get_all_tickers(db_info)

ticker_dg = 'A' + ticker
revenue_dg = fs_df[(fs_df['symbol'] == ticker_dg) & (fs_df['indicator'] == '매출액(천원)')]
revenue_from_dg = revenue_dg[['date', 'value']].copy()
revenue_from_dg['value'] = revenue_from_dg['value'] * 1000

# 분기별 매출
print("=" * 80)
print(f"{ticker} 순수 분기별 매출")
print("=" * 80)

revenue_df = get_quarterly_revenue_simple(db_info, ticker=ticker)

if not revenue_df.empty:
    recent = revenue_df.tail(12)
    print("\n최근 12개 분기:")
    print(recent[['bsns_year', 'quarter', 'report_date', 'thstrm_amount']].to_string(index=False))

    print("\n최근 8개 분기 (억원):")
    summary = revenue_df.tail(8)[['bsns_year', 'quarter', 'report_date', 'thstrm_amount']].copy()
    summary['매출_억원'] = (summary['thstrm_amount'] / 100_000_000).round(0).astype(int)
    print(summary[['bsns_year', 'quarter', 'report_date', '매출_억원']].to_string(index=False))

revenue_from_dart = revenue_df[['report_date', 'thstrm_amount']].copy()
revenue_from_dg.columns = ['date', 'revenue']
revenue_from_dart.columns = ['date', 'revenue']

# 데이터 결합
revenue_concated_df = pd.concat([revenue_from_dg, revenue_from_dart], axis=0)\
    .drop_duplicates(subset=['date'], keep='first')\
    .sort_values('date')\
    .reset_index(drop=True)

# ---- 1) 데이터 준비 - 분기 인덱스로 변환 ------------------------------------

df = revenue_concated_df.copy()

# datetime으로 변환
df['date'] = pd.to_datetime(df['date'])

# 분기 정보 추출
df['year'] = df['date'].dt.year
df['quarter'] = df['date'].dt.quarter

# 분기 PeriodIndex 생성 (이게 핵심!)
df['period'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)
df['period'] = pd.PeriodIndex(df['period'], freq='Q')

print("\n데이터 샘플 (분기 변환):")
print(df[['date', 'year', 'quarter', 'period', 'revenue']].tail(10))

# 중복된 분기가 있는지 확인
duplicates = df[df.duplicated(subset=['period'], keep=False)]
if not duplicates.empty:
    print(f"\n⚠️ 중복된 분기 발견: {len(duplicates)}개")
    print(duplicates[['date', 'period', 'revenue']])
    # 중복 처리: 각 분기의 마지막 값 사용
    df = df.sort_values(['period', 'date']).groupby('period').last().reset_index()

# PeriodIndex를 인덱스로 설정
df = df.set_index('period').sort_index()
series = df['revenue'].astype(float)

print(f"\n시계열 정보:")
print(f"  - 시작: {series.index[0]}")
print(f"  - 종료: {series.index[-1]}")
print(f"  - 빈도: {series.index.freq}")
print(f"  - 데이터 개수: {len(series)}")

# 최소 데이터 길이 검사
n = len(series)
if n < 16:
    raise ValueError(f"❌ 데이터가 {n}개입니다. 최소 16개 분기 이상 필요합니다.")
else:
    print(f"✅ 데이터 개수 OK: {n}개")

# ---- 2) 빈도 설정 ---------------------------------------------------------

m = 4  # 분기 데이터의 계절성은 4

print(f"\n계절성 파라미터: m = {m}")

# ---- 3) 모델별 예측 (9분기) -----------------------------------------------

print("\n=== SARIMA 예측 중 ===")
sarima_result = forecast_sarima(
    y=series,
    forecast_horizon=H,
    seasonal_period=m,
    try_transforms=True
)

print("\n=== ETS 예측 중 ===")
ets_result = forecast_ets(
    y=series,
    forecast_horizon=H,
    m=m,
    try_transforms=True
)

print("\n=== Theta 예측 중 ===")
theta_result = forecast_theta(
    y=series,
    forecast_horizon=H,
    m=m,
    try_transforms=True
)

# ---- 4) 결과 합치기 - PeriodIndex로 생성 -----------------------------------

# 마지막 분기 기준으로 미래 분기 생성
last_period = series.index[-1]

# 미래 9분기 생성
forecast_periods = pd.period_range(
    start=last_period + 1,
    periods=H,
    freq='Q'
)

print(f"\n예측 분기:")
print(f"  시작: {forecast_periods[0]}")
print(f"  종료: {forecast_periods[-1]}")
print(f"  개수: {len(forecast_periods)}")
print(f"\n전체 예측 분기:")
for i, p in enumerate(forecast_periods, 1):
    print(f"  {i}. {p}")

result_df = pd.DataFrame({
    "SARIMA": sarima_result.get("forecast"),
    "ETS": ets_result.get("forecast"),
    "Theta": theta_result.get("forecast"),
}, index=forecast_periods)

print("\n=== 최종 예측 결과 ===")
print(result_df)

# 앙상블(세 모델 평균) 컬럼 추가
result_df["Ensemble"] = result_df[["SARIMA", "ETS", "Theta"]].mean(axis=1)
result_df['ticker'] = ticker

# 억원 단위로 변환하여 보기 쉽게 출력
print("\n=== 예측 결과 (억원) ===")
result_display = result_df.copy()
for col in ["SARIMA", "ETS", "Theta", "Ensemble"]:
    result_display[f"{col}_억원"] = (result_display[col] / 100_000_000).round(0).astype(int)

display_cols = ['ticker'] + [col for col in result_display.columns if '억원' in col]
print(result_display[display_cols])

# DatetimeIndex로 변환하여 저장하고 싶다면
result_df_with_date = result_df.copy()
result_df_with_date.index = result_df_with_date.index.to_timestamp()

print("\n✅ 예측 완료!")
print(f"\n최종 결과:")
print(f"  - 예측 분기 수: {len(result_df)}개")
print(f"  - 예측 기간: {forecast_periods[0]} ~ {forecast_periods[-1]}")

✅ unique ticker 2451개 조회 완료
131290 순수 분기별 매출
✅ 131290 매출 데이터 40행 조회
2015년: FY(132,224,416,776.0) - Q1+Q2+Q3(0.0) = Q4(132,224,416,776.0)
2016년: FY(126,712,202,718.0) - Q1+Q2+Q3(93,696,148,872.0) = Q4(33,016,053,846.0)
2017년: FY(185,749,435,431.0) - Q1+Q2+Q3(142,714,008,761.0) = Q4(43,035,426,670.0)
2018년: FY(184,431,173,453.0) - Q1+Q2+Q3(132,540,316,627.0) = Q4(51,890,856,826.0)
2019년: FY(191,490,313,556.0) - Q1+Q2+Q3(127,431,748,028.0) = Q4(64,058,565,528.0)
2020년: FY(285,508,585,365.0) - Q1+Q2+Q3(223,068,763,761.0) = Q4(62,439,821,604.0)
2021년: FY(307,663,254,437.0) - Q1+Q2+Q3(220,331,341,048.0) = Q4(87,331,913,389.0)
2022년: FY(339,264,169,508.0) - Q1+Q2+Q3(257,671,317,855.0) = Q4(81,592,851,653.0)
2023년: FY(249,147,689,845.0) - Q1+Q2+Q3(178,869,644,106.0) = Q4(70,278,045,739.0)
2024년: FY(348,053,207,965.0) - Q1+Q2+Q3(244,935,593,401.0) = Q4(103,117,614,564.0)

최근 12개 분기:
 bsns_year quarter report_date  thstrm_amount
      2022      Q4  2022-12-31   8.159285e+10
      2023      Q1  2

In [16]:
result_df

,SARIMA,ETS,Theta,Ensemble,ticker
2025Q4,9.745671e+10,1.546865e+11,1.012260e+11,1.177897e+11,131290
2026Q1,8.889455e+10,1.471317e+11,9.940497e+10,1.118104e+11,131290
2026Q2,1.036620e+11,1.346490e+11,1.055107e+11,1.146072e+11,131290
2026Q3,1.102345e+11,1.549365e+11,1.106718e+11,1.252809e+11,131290
2026Q4,1.053584e+11,1.649454e+11,1.039412e+11,1.247483e+11,131290
2027Q1,9.504226e+10,1.573906e+11,1.021202e+11,1.181844e+11,131290
2027Q2,1.098097e+11,1.449080e+11,1.082259e+11,1.209812e+11,131290
2027Q3,1.163822e+11,1.651955e+11,1.133870e+11,1.316549e+11,131290
2027Q4,1.115061e+11,1.752043e+11,1.066564e+11,1.311223e+11,131290


In [17]:
"""
단일 ticker 매출 예측 테스트 스크립트
"""

import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

# ==================== 경로 설정 ====================
def setup_paths():
    current = Path.cwd()
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            if str(data_folder) not in sys.path:
                sys.path.insert(0, str(data_folder))
            print(f"✅ 경로 설정 완료: {parent}")
            return parent
    return current

setup_paths()

# ==================== Import ====================
from universal_ts_forecast_function import (
    forecast_sarima,
    forecast_ets,
    forecast_theta
)
from DATA.stock_invest_function import get_db_host
import pymysql

# ==================== DB 설정 ====================
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# ==================== 함수 정의 ====================

def get_quarterly_revenue_simple(db_info: dict, ticker: str, adjust_q4: bool = True):
    """DART에서 매출 데이터 조회"""
    ticker_str = str(ticker).zfill(6)

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],
        charset="utf8mb4",
    )

    try:
        sql = """
            SELECT *
            FROM korea_fs_data_from_DART
            WHERE ticker = %s
              AND account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
            ORDER BY bsns_year, report_date
        """
        df = pd.read_sql(sql, conn, params=[ticker_str])

        if df.empty:
            return df

        if adjust_q4:
            df = adjust_fy_to_q4(df)

        return df
    finally:
        conn.close()


def adjust_fy_to_q4(df: pd.DataFrame) -> pd.DataFrame:
    """FY를 순수 Q4로 변환"""
    result_df = df.copy()

    for year in result_df['bsns_year'].unique():
        year_mask = result_df['bsns_year'] == year
        fy_mask = year_mask & (result_df['quarter'] == 'FY')

        if fy_mask.any():
            fy_amount = result_df.loc[fy_mask, 'thstrm_amount'].iloc[0]
            q123_mask = year_mask & result_df['quarter'].isin(['Q1', 'Q2', 'Q3'])
            q123_sum = result_df.loc[q123_mask, 'thstrm_amount'].sum()
            pure_q4 = fy_amount - q123_sum

            result_df.loc[fy_mask, 'quarter'] = 'Q4'
            result_df.loc[fy_mask, 'thstrm_amount'] = pure_q4

    return result_df


def prepare_quarterly_series(revenue_df: pd.DataFrame, ticker: str) -> pd.Series:
    """매출 DataFrame을 분기별 PeriodIndex Series로 변환"""
    df = revenue_df.copy()

    df['report_date'] = pd.to_datetime(df['report_date'])
    df['year'] = df['report_date'].dt.year
    df['quarter'] = df['report_date'].dt.quarter

    # 분기 PeriodIndex 생성
    df['period'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)
    df['period'] = pd.PeriodIndex(df['period'], freq='Q')

    # 중복 처리
    duplicates = df[df.duplicated(subset=['period'], keep=False)]
    if not duplicates.empty:
        df = df.sort_values(['period', 'report_date']).groupby('period').last().reset_index()

    df = df.set_index('period').sort_index()
    series = df['thstrm_amount'].astype(float)

    return series


# ==================== 단일 Ticker 예측 함수 ====================

def test_single_ticker(ticker: str, H: int = 9):
    """
    단일 ticker 매출 예측 테스트

    Parameters:
    -----------
    ticker : str
        종목 코드 (예: "005930", "131290")
    H : int
        예측 분기 수 (기본: 9)
    """

    print("=" * 80)
    print(f"🔍 Ticker {ticker} 매출 예측 테스트")
    print("=" * 80)

    try:
        # 1) 데이터 로드
        print("\n[1단계] 데이터 로드 중...")
        revenue_df = get_quarterly_revenue_simple(db_info, ticker=ticker)

        if revenue_df.empty:
            print(f"❌ ticker {ticker}의 데이터가 없습니다")
            return None

        print(f"✅ 데이터 로드 완료: {len(revenue_df)}개 레코드")

        # 최근 데이터 확인
        print("\n[최근 12개 분기 데이터]")
        recent = revenue_df.tail(12)[['bsns_year', 'quarter', 'report_date', 'thstrm_amount']].copy()
        recent['매출_억원'] = (recent['thstrm_amount'] / 100_000_000).round(0).astype(int)
        print(recent[['bsns_year', 'quarter', 'report_date', '매출_억원']].to_string(index=False))

        # 2) 분기 Series로 변환
        print("\n[2단계] 분기 데이터로 변환 중...")
        series = prepare_quarterly_series(revenue_df, ticker)

        print(f"✅ 변환 완료")
        print(f"   - 시작 분기: {series.index[0]}")
        print(f"   - 종료 분기: {series.index[-1]}")
        print(f"   - 총 분기 수: {len(series)}")
        print(f"   - 빈도: {series.index.freq}")

        # 최소 데이터 체크
        if len(series) < 16:
            print(f"⚠️ 경고: 데이터가 {len(series)}개 분기입니다. 16개 이상 권장")

        # 3) 예측 실행
        print("\n[3단계] 모델 예측 중...")
        m = 4  # 분기 계절성

        print("  - SARIMA 예측 중...")
        sarima_result = forecast_sarima(
            y=series,
            forecast_horizon=H,
            seasonal_period=m,
            try_transforms=True
        )

        print("  - ETS 예측 중...")
        ets_result = forecast_ets(
            y=series,
            forecast_horizon=H,
            m=m,
            try_transforms=True
        )

        print("  - Theta 예측 중...")
        theta_result = forecast_theta(
            y=series,
            forecast_horizon=H,
            m=m,
            try_transforms=True
        )

        print("✅ 모델 예측 완료")

        # 4) 결과 정리
        print("\n[4단계] 결과 정리 중...")
        last_period = series.index[-1]

        # 미래 분기 생성
        forecast_periods = pd.period_range(
            start=last_period + 1,
            periods=H,
            freq='Q'
        )

        result_df = pd.DataFrame({
            "SARIMA": sarima_result.get("forecast"),
            "ETS": ets_result.get("forecast"),
            "Theta": theta_result.get("forecast"),
        }, index=forecast_periods)

        # 앙상블
        result_df["Ensemble"] = result_df[["SARIMA", "ETS", "Theta"]].mean(axis=1)

        print("✅ 결과 정리 완료")

        # 5) 결과 출력
        print("\n" + "=" * 80)
        print("📊 예측 결과")
        print("=" * 80)

        print(f"\n예측 기간: {forecast_periods[0]} ~ {forecast_periods[-1]}")
        print(f"예측 분기 수: {len(forecast_periods)}개\n")

        # 원 단위 결과
        print("[원 단위 예측값]")
        print(result_df)

        # 억원 단위 결과
        print("\n[억원 단위 예측값]")
        result_display = result_df.copy()
        for col in ["SARIMA", "ETS", "Theta", "Ensemble"]:
            result_display[f"{col}_억원"] = (result_display[col] / 100_000_000).round(0).astype(int)

        display_cols = [col for col in result_display.columns if '억원' in col]
        print(result_display[display_cols])

        # 6) CSV 저장
        output_file = f'revenue_forecast_{ticker}.csv'
        result_df_save = result_df.copy()
        result_df_save['ticker'] = ticker
        result_df_save['date'] = result_df_save.index.to_timestamp()
        result_df_save.reset_index(drop=True, inplace=True)
        result_df_save.to_csv(output_file, index=False, encoding='utf-8-sig')

        print(f"\n✅ 결과 저장 완료: {output_file}")

        return result_df

    except Exception as e:
        print(f"\n❌ 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        return None


# ==================== 실행 ====================

if __name__ == "__main__":

    # 테스트할 ticker 입력
    # 예시: "005930" (삼성전자), "131290" (티에스이), "000660" (SK하이닉스)

    test_ticker = "131290"  # 여기에 원하는 ticker 입력

    result = test_single_ticker(ticker=test_ticker, H=9)

    if result is not None:
        print("\n" + "=" * 80)
        print("✅ 테스트 완료!")
        print("=" * 80)

✅ 경로 설정 완료: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
🔍 Ticker 131290 매출 예측 테스트

[1단계] 데이터 로드 중...
✅ 데이터 로드 완료: 40개 레코드

[최근 12개 분기 데이터]
 bsns_year quarter report_date  매출_억원
      2022      Q4  2022-12-31   1689
      2023      Q1  2023-03-31    446
      2023      H1  2023-06-30    573
      2023      Q3  2023-09-30    769
      2023      Q4  2023-12-31   1276
      2024      Q1  2024-03-31    582
      2024      H1  2024-06-30    760
      2024      Q3  2024-09-30   1107
      2024      Q4  2024-12-31   1792
      2025      Q1  2025-03-31    830
      2025      H1  2025-06-30   1176
      2025      Q3  2025-09-30   1044

[2단계] 분기 데이터로 변환 중...
✅ 변환 완료
   - 시작 분기: 2015Q4
   - 종료 분기: 2025Q3
   - 총 분기 수: 40
   - 빈도: <QuarterEnd: startingMonth=12>

[3단계] 모델 예측 중...
  - SARIMA 예측 중...
[메모리] forecast_sarima 실행 전: 4728.55 MB
[메모리] find_best_sarima_params 실행 전: 4728.55 MB
[메모리] find_best_sarima_params 실행 후: 4728.61 MB (변화: +0.05 MB)
[메모리] forecast_sarima 실행 후: 4728.61 MB (

In [18]:
result

,SARIMA,ETS,Theta,Ensemble
2025Q4,1.875008e+11,1.737461e+11,2.013829e+11,1.875433e+11
2026Q1,8.830184e+10,9.160338e+10,9.050942e+10,9.013821e+10
2026Q2,1.127448e+11,1.100409e+11,1.009423e+11,1.079094e+11
2026Q3,1.176635e+11,1.188842e+11,1.136675e+11,1.167384e+11
2026Q4,1.920003e+11,1.799094e+11,2.074032e+11,1.931043e+11
2027Q1,9.736079e+10,9.776663e+10,9.319508e+10,9.610750e+10
2027Q2,1.218038e+11,1.162042e+11,1.039155e+11,1.139745e+11
2027Q3,1.267224e+11,1.250475e+11,1.169910e+11,1.229203e+11
2027Q4,2.010593e+11,1.860726e+11,2.134234e+11,2.001851e+11


In [22]:
from Korea_revenue_forecast_batch_with_input_date import batch_forecast_all_tickers

result_df, error_list = batch_forecast_all_tickers(
    db_info=db_info,
    fs_df=fs_df,
    input_date='2024-12-09',
    H=9,
    ticker_start_idx=0,    # 0번째부터
    ticker_end_idx=50,     # 49번째까지 (50개)
    save_to_db=True
)

ModuleNotFoundError: No module named 'Korea_revenue_forecast_batch_with_input_date'